# Assignment 20 – Recommendation System

In [1]:
import pandas as pd
# load dataset (place movies_metadata.csv in folder)
df = pd.read_csv('movies_metadata.csv', low_memory=False)
df = df[['title','overview']].dropna().head(2000)
df.head()

,title,overview
0,Toy Story,"Led by Woody, Andy's toys live happily in his ..."
1,Jumanji,When siblings Judy and Peter discover an encha...
2,Grumpier Old Men,A family wedding reignites the ancient feud be...
3,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom..."
4,Father of the Bride Part II,Just when George Banks has recovered from his ...


In [2]:
import re
from nltk.corpus import stopwords
import nltk
nltk.download('stopwords')
stop = set(stopwords.words('english'))

def clean(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    words = [w for w in text.split() if w not in stop]
    return ' '.join(words)

df['clean_text'] = df['overview'].apply(clean)
df.head()

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/harshgarg/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,title,overview,clean_text
0,Toy Story,"Led by Woody, Andy's toys live happily in his ...",led woody andys toys live happily room andys b...
1,Jumanji,When siblings Judy and Peter discover an encha...,siblings judy peter discover enchanted board g...
2,Grumpier Old Men,A family wedding reignites the ancient feud be...,family wedding reignites ancient feud nextdoor...
3,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",cheated mistreated stepped women holding breat...
4,Father of the Bride Part II,Just when George Banks has recovered from his ...,george banks recovered daughters wedding recei...


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
matrix = tfidf.fit_transform(df['clean_text'])
print(matrix.shape)

(2000, 5000)


In [4]:
from sklearn.metrics.pairwise import cosine_similarity
similarity = cosine_similarity(matrix)

In [5]:
def recommend(title):
    if title not in df['title'].values:
        return 'Movie not found'
    idx = df[df['title']==title].index[0]
    scores = list(enumerate(similarity[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)[1:6]
    return [df.iloc[i[0]].title for i in scores]

print(recommend('Toy Story'))

['Condorman', 'Rebel Without a Cause', 'Malice', 'The Sunchaser', 'Pretty Woman']
